**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**26 | Final Development Model Training, Reproducible Inference and Qualified Test Gate**

![Stage 26 scientific workflow](../images/26_workflow.svg)

**Objective:** Fit the final 1D-ResNet and Random Forest reference on all 29 HARTH Development participants, save and reload the fitted models, and report the separately approved qualified held-out evaluation.

**Dataset:** Pinned HARTH v2.0 recordings with six acceleration channels at 50 Hz, 100-sample label-pure windows, a 50-sample stride and a 500-window-per-participant cap. The 29-person Development cohort contains 14,201 selected windows.

**Notebook Scope:** Sections 01–10 fit models and verify Development-only inference without accessing reserved Test recordings. Sections 11–13 contain the separately authorised, one-time evaluation of 500 windows each from S023 and S032, using the unchanged Development-fitted models.

**Approval and evidence boundary:** The Stage 25 candidate manifest was still marked `PENDING_USER_REVIEW` during the initial Development-only sections. A separate approved manifest was subsequently verified before Test access. Both reserved participants had been inspected descriptively earlier, so the saved results are qualified same-study held-out evidence, not a strictly blind or external evaluation.

In [ ]:
%cd /content

!git clone https://github.com/RezaSaadatyar/human-movement-intelligence.git

%cd /content/human-movement-intelligence

from pathlib import Path

root = Path.cwd().resolve()

required_file = root / "code/python/src/core/data_loading.py"

print("Project root:", root)
print("Required file exists:", required_file.is_file())

**01. Imports and Final Research Environment**

In [ ]:
# ================ 01. Imports and Final Research Environment =================

# ----------------- 1. Import Python standard-library modules -----------------
import sys
import json
from pathlib import Path

# -------------- 2. Import numerical and tabular analysis tools ---------------
import numpy as np
import pandas as pd

# --------------- 3. Import plotting and notebook display tools ---------------
import matplotlib.pyplot as plt
from IPython.display import display

# --------------- 4. Configure portable repository import paths ---------------
root=Path.cwd().resolve()
while not (root/"code/python/src/core/harth_final_inference.py").is_file():
    if root.parent==root:
        raise RuntimeError("Run within the HMI repository.")
    root=root.parent
sys.path.insert(0,str(root/"code/python/src"))
sys.path.insert(0,str(root/"studies/wearable_activity_recognition/src"))

# --------------- 5. Import final-model and approval utilities ----------------
from core.harth_final_inference import (
    load_final_protocol, fit_final_resnet, fit_development_scaler,
    apply_frozen_scaler, save_final_models, load_final_models,
    infer_source_windows, enforce_qualified_test_approval,
    score_qualified_heldout,
)

# -------------- 6. Import HARTH source and channel definitions ---------------
from harth_data import (
    HARTH_SUBJECTS, SENSOR_COLUMNS, HARTH_REVISION, ACTIVITY_NAMES,
    prepare_harth_window_cohort,
)

# ----------------- 7. Import statistical feature extraction ------------------
from core.feature_engineering import extract_signal_features

# ------------------- 8. Import the Random Forest estimator -------------------
from sklearn.ensemble import RandomForestClassifier

# ----------------- 9. Import PyTorch for the final model fit -----------------
import torch

# ----------- 10. Load candidate protocol without authorising Test ------------
PROPOSAL=root/"studies/wearable_activity_recognition/research_outputs/stage_25_protocol_proposal_reconciled.json"
protocol=load_final_protocol(PROPOSAL,root,require_approval=False)
RESERVED_TEST={"S023","S032"}
DEVELOPMENT=sorted(set(HARTH_SUBJECTS)-RESERVED_TEST)
CLASSES=tuple(protocol["source"]["task_codes"])
ARTIFACT_DIR=root/"outputs/harth_stage26_final"
assert len(DEVELOPMENT)==29 and HARTH_REVISION==protocol["source"]["revision"]
assert len(SENSOR_COLUMNS)==6

# ----------- 11. Require CUDA and record the training environment ------------
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Colab: Runtime > Change runtime type > GPU.")
DEVICE="cuda"
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
print(""+"="*70)
print("STAGE 26 | SAFE FINAL DEVELOPMENT MODEL FIT")
print("="*70)
print("GPU:",torch.cuda.get_device_name(0),"| PyTorch:",torch.__version__)
print("Source revision:",HARTH_REVISION)
print("Protocol approval status:",protocol["status"])
print("Reserved Test loaded: NO")
print("="*70)

**02. Original 29-Person Development Cohort and Source Integrity**

Construct the exact source-row-aligned 500-cap cohort from the official pinned recordings, applying strict time continuity and label-purity checks **before** filtering out-of-task annotations. Do not use `S023` or `S032` for fitting, feature extraction, batch-normalisation statistics, class weights or architecture decisions. The manifest's Development SHA256 is compared with the exact reconstructed sample used in Stages 23–25.

In [ ]:
# ====================== 02. Verified Development Cohort ======================

# ------------ 1. Recreate the bounded Development source windows -------------
X_dev,y_dev,dev_metadata,dev_overview=prepare_harth_window_cohort(
    subjects=DEVELOPMENT,data_dir=root/"data/raw/harth_v2",
    task_classes=CLASSES,max_windows_per_subject=500,
    window_size=100,step_size=50,sampling_rate=50,
)
assert X_dev.shape==(14201,6,100)
assert len(y_dev)==len(dev_metadata)
assert set(dev_metadata["subject_id"])==set(DEVELOPMENT)
assert not(set(dev_metadata["subject_id"])&RESERVED_TEST)

# ---------------- 2. Import the shared fingerprinting utility ----------------
from core.temporal_benchmark import make_benchmark_plan

# --------------- 3. Verify the Development source fingerprint ----------------
dev_plan=make_benchmark_plan(
    X_dev,y_dev,dev_metadata["subject_id"].to_numpy(),
    classes=CLASSES,n_splits=3,forbidden_subjects=RESERVED_TEST,
)
assert dev_plan["fingerprint"]==protocol["source_development_sha256"]
display(dev_overview[["subject_id","eligible_windows","sampled_windows"]])
print("Development SHA256 verified:",dev_plan["fingerprint"])
print("Actual Test source files opened: NO")

**03. Development-Only Channel Scaling and Source Class Weights**

The six original sensor axes must remain in the same order in every training and inference window. Channel statistics are calculated from *all final Development windows only*, not from the protected Test. Weighted Cross-Entropy receives Train-only class weights based on actual original activity codes. These parameters are not tuned using held-out observations.

In [ ]:
# ==================== 03. Development-Only Preprocessing =====================

# --------------- 1. Fit channel scaling using Development only ---------------
scaler=fit_development_scaler(X_dev)
X_scaled=apply_frozen_scaler(X_dev,scaler)
assert X_scaled.shape==X_dev.shape and np.isfinite(X_scaled).all()
stats=pd.DataFrame({
    "channel":SENSOR_COLUMNS,
    "development_mean":scaler["mean"],
    "development_std":scaler["std"],
})
display(stats.round(6))

# ---------------- 2. Report the Development activity support -----------------
source_counts=pd.Series(y_dev).value_counts().reindex(CLASSES,fill_value=0)
display(pd.DataFrame({
    "source_code":CLASSES,
    "activity":[ACTIVITY_NAMES[c] for c in CLASSES],
    "development_windows":source_counts.to_numpy(),
}))
assert (source_counts>0).all()

**04. Train the Final 1D-ResNet Using the Frozen 400-Update Procedure**

A newly initialised 1D-ResNet trains on the entire eligible Development cohort using seed 42, batch size 128, Adam 0.001 and inverse-square-root class weighting. It is deliberately trained for exactly 400 optimiser updates and the last-step weights are retained. There is no Validation or Test-driven checkpoint selection. The training loss trace is a diagnostic, **not** an unbiased accuracy estimate.

In [ ]:
# ====================== 04. Final Neural Model Training ======================

# ------------------- 1. Train the fixed 400-update ResNet --------------------
model,trained_scaler,train_history,class_weights,train_time=fit_final_resnet(
    X_dev,y_dev,protocol,device=DEVICE
)

# ------------- 2. Check the training budget and plot Train loss --------------
assert int(train_history["step"].iloc[-1])==400
display(train_history)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(train_history["step"],train_history["development_train_loss"],marker="o")
ax.set(title="Final 1D-ResNet | Development Train loss only",
       xlabel="Optimizer update",ylabel="Weighted Cross-Entropy loss")
ax.grid(alpha=.2)
fig.tight_layout()
plt.show()
print("Final model fit duration (seconds):",round(train_time,2))
print("Test data used in training: NO")

**05. Fit the Frozen Random Forest as a Classical Reference**

The Random Forest reference uses 30 deterministic features from the same eligible Development source windows. Feature extraction independently summarises each six-channel window and does not fit a Test-derived transform. The 120-tree specification is frozen from Stage 25, not selected after seeing Test performance.

In [ ]:
# ===================== 05. Fixed Random Forest Reference =====================

# ------------- 1. Extract 30 ordered features per source window --------------
features=extract_signal_features(X_dev,channel_names=SENSOR_COLUMNS)
assert features.shape==(14201,30)

# ------------- 2. Fit the 120-tree Random Forest on Development --------------
rf=RandomForestClassifier(
    n_estimators=120,min_samples_leaf=3,
    class_weight="balanced_subsample",random_state=42,n_jobs=-1
)
rf.fit(features.to_numpy(),y_dev)
assert set(rf.classes_)==set(CLASSES)
print("RF trees:",len(rf.estimators_))
print("RF feature count:",rf.n_features_in_)
print("RF Train subjects:",len(DEVELOPMENT))
print("RF Test subjects accessed: NO")

**06. Save Model Artefacts and Inference Metadata**

Save the ResNet weights, Development-fitted channel statistics, Random Forest reference, original class mapping, feature-column order and source/protocol fingerprints. Model artefacts must be reloaded only from a trusted local directory; they do not contain the original Test labels or recordings.

In [ ]:
# ======================== 06. Persist Model Artefacts ========================

# --------------- 1. Save fitted models and inference metadata ----------------
artifact_info=save_final_models(
    model,trained_scaler,rf,list(features.columns),
    protocol,dev_plan["fingerprint"],ARTIFACT_DIR
)
display(pd.DataFrame([
    {"artifact":name,"path":path,"sha256":artifact_info["sha256"][name]}
    for name,path in artifact_info.items() if name!="sha256"
]))
print("Model artefacts stored locally:",ARTIFACT_DIR)
print("Artefacts are ignored by Git and should not be committed.")

**07. Reload Models and Verify Development Predictions**

Reload both saved models and their preprocessing configuration, then run inference on 24 Development windows. Confirm that the RF predictions match the in-memory reference and that both reloaded models return valid source activity codes. This is an engineering round-trip check, not an evaluation of performance on unseen participants.

In [ ]:
# ===================== 07. Reload and Verify Predictions =====================

# ------------- 1. Reload models from the trusted local directory -------------
loaded_model,loaded_rf,config=load_final_models(ARTIFACT_DIR,device="cpu")
N_CHECK=24
unseen_style_windows=X_dev[:N_CHECK]
reloaded_nn,reloaded_rf=infer_source_windows(
    unseen_style_windows,loaded_model,loaded_rf,config,device="cpu"
)

# ------------ 2. Check source codes and RF prediction consistency ------------
assert len(reloaded_nn)==len(reloaded_rf)==N_CHECK
assert set(reloaded_nn).issubset(CLASSES)
assert set(reloaded_rf).issubset(CLASSES)
direct_rf=rf.predict(
    features.iloc[:N_CHECK].to_numpy()
)
np.testing.assert_array_equal(reloaded_rf,direct_rf)
display(pd.DataFrame({
    "true_dev_code":y_dev[:N_CHECK],
    "reloaded_resnet_prediction":reloaded_nn,
    "reloaded_rf_prediction":reloaded_rf,
}))
print("Saved neural and classical inference: PASSED")
print("Independent Test model performance assessed: NO")

**08. Approval Gate Without Test Access**

This earlier Development-only checkpoint reads the Stage 25 proposal, which was marked `PENDING_USER_REVIEW`, and does not load or score S023 or S032. Its saved output documents the initial blocked state. The separately approved evaluation in Section 11 follows a different, explicit approval check.

In [ ]:
# =================== 08. Approval Gate Without Test Access ===================

# --------------- 1. Report the initial proposal approval state ---------------
print("Protocol status:",protocol["status"])
print("Qualified Test requested in this notebook: NO")
if protocol["status"]!="APPROVED":
    print("Qualified Test gate: BLOCKED pending explicit user review.")
else:
    print("Qualified Test gate: approval recorded; separate confirmation still required.")

# -------------- 2. Confirm the reserved Test identity boundary ---------------
assert RESERVED_TEST=={"S023","S032"}
print("S023 and S032 recordings read in this notebook: NO")

**09. Qualified Test Reporting Contract**

Before the separately authorised evaluation, specify the fixed eight-class Macro F1, secondary accuracy metrics, per-class precision/recall/F1 and support, participant-level reporting and confusion matrices. This section defines the reporting requirements; it does not calculate Test scores.

In [ ]:
# =================== 09. Qualified Test Reporting Contract ===================

# ------------- 1. Document the predefined qualified Test metrics -------------
report_plan=pd.DataFrame([
    {"evaluation":"Primary metric","requirement":"Fixed eight-class Macro F1"},
    {"evaluation":"Secondary metrics","requirement":"Balanced accuracy and overall accuracy"},
    {"evaluation":"Per-class diagnostics","requirement":"Precision, recall, F1, actual support"},
    {"evaluation":"Independent people","requirement":"S023 and S032 reported separately"},
    {"evaluation":"Model comparison","requirement":"Frozen ResNet and predeclared RF reference"},
    {"evaluation":"Forbidden operations","requirement":"No Test-driven refitting, model choice or hyperparameter search"},
    {"evaluation":"Scientific limitation","requirement":"Originally descriptively inspected; same-study, not external"},
])
display(report_plan)
print("Test metrics created by this stage: NONE")

**10. Development Fitting Quality Gate**

Verify the 29-person Development cohort, source fingerprints, final ResNet training budget, RF configuration, saved metadata and trusted inference round-trip. At this checkpoint, the reserved recordings have not been opened. Test approval and evaluation are separate, subsequent steps.

In [ ]:
# =================== 10. Development Fitting Quality Gate ====================

# ------------- 1. Verify the Development cohort and final models -------------
assert X_dev.shape==(14201,6,100)
assert set(dev_metadata["subject_id"])==set(DEVELOPMENT)
assert not(set(dev_metadata["subject_id"])&RESERVED_TEST)
assert protocol["primary_model"]["optimizer_steps"]==400
assert train_history["step"].iloc[-1]==400
assert len(class_weights)==8 and np.isfinite(class_weights).all()
assert all(Path(path).is_file() for name,path in artifact_info.items() if name!="sha256")

# --------------- 2. Check saved preprocessing and RF inference ---------------
assert config["source_codes"]==list(CLASSES)
assert config["sensor_columns"]==SENSOR_COLUMNS
assert config["preprocessing_fit_scope"]=="Development only"
assert not config["test_fitted_values"]
np.testing.assert_array_equal(reloaded_rf,direct_rf)

# --------------- 3. Report Development fitting and gate status ---------------
print(""+"="*70)
print("STAGE 26 | FROZEN DEVELOPMENT FIT AND INFERENCE ARTEFACTS")
print("="*70)
print("Frozen 29-person Development cohort: VERIFIED")
print("Primary 1D-ResNet trained: YES, 400 fixed updates")
print("Fixed Random Forest trained: YES, 120 trees")
print("Train-only scaler, class map and channel order: SAVED")
print("Trusted artifact reloading/inference: PASSED")
print("Original S023/S032 Test accessed: NO")
print("-"*70)
print("Stage 26 Development training/inference quality gate: PASSED")
print("Protocol approval status:",protocol["status"])
print("Qualified Test evaluation performed: NO")
print("Next: explicit qualified-Test approval and separate gated scoring")
print("="*70)

**11. Approved One-Time Qualified Test Inference**

This saved section records the separately approved evaluation under the unchanged Stage 25 protocol. Before accessing S023 or S032, the code checks the approved manifest against the candidate protocol and verifies the previously saved Development-fitted artefacts.

Exactly 500 eligible windows from each reserved participant are prepared with the fixed settings and scored by both frozen models. Neither Test labels nor Test metrics are used to update preprocessing, weights, hyperparameters, thresholds or model choice. Do not rerun this evaluation for model selection.

In [ ]:
# ============== 11. Approved One-Time Qualified Test Inference ===============

# ------------- 1. Import standard-library hashing and JSON tools -------------
import hashlib
import json

# ------------- 2. Import protocol approval and frozen inference --------------
from core.harth_final_inference import (
    sha256_file, load_final_protocol, enforce_qualified_test_approval,
    load_final_models, infer_source_windows, TEST_CONFIRMATION,
)

# --------------- 3. Import qualified evaluation and reporting ----------------
from core.harth_qualified_test_report import (
    validate_final_test_inputs, summarise_qualified_heldout,
    SOURCE_CLASSES, RESERVED_PEOPLE, FINAL_MODELS,
)

# ------------- 4. Verify separate approval against the candidate -------------
approved_path = root / "studies/wearable_activity_recognition/research_outputs/stage_25_protocol_approved_qualified_test.json"
approved = load_final_protocol(approved_path, root, require_approval=True)
assert approved["approval_record"]["scope"].startswith("One-time qualified")
assert approved["source_development_sha256"] == protocol["source_development_sha256"]
assert approved["primary_model"] == protocol["primary_model"]
assert approved["reference_model"] == protocol["reference_model"]
assert approved["source"] == protocol["source"]
assert approved["evaluation"] == protocol["evaluation"]
assert approved["module_sha256"] == protocol["module_sha256"]
assert enforce_qualified_test_approval(approved, TEST_CONFIRMATION)

# ----------- 5. Validate and reload trusted Development artefacts ------------
expected_artifacts = ["resnet_state_dict.pt", "random_forest.joblib", "inference_config.json"]
for filename in expected_artifacts:
    assert (ARTIFACT_DIR / filename).is_file(), (
        f"Missing Development artefact {filename}; recover trusted files or "
        "rerun the unchanged Development-only Sections 01–10."
    )
assert approved["source_development_sha256"] == dev_plan["fingerprint"]
assert config["development_cohort_sha256"] == approved["source_development_sha256"]
assert config["sensor_columns"] == SENSOR_COLUMNS
assert config["source_codes"] == list(SOURCE_CLASSES)
assert config["preprocessing_fit_scope"] == "Development only"
assert not config["test_fitted_values"]
assert all(sha256_file(artifact_info[key]) == artifact_info["sha256"][key]
           for key in ("resnet", "reference", "config"))
frozen_nn, frozen_rf, frozen_config = load_final_models(ARTIFACT_DIR, device="cpu")
assert frozen_config == config
print("Approved protocol and trusted Development artefacts: VERIFIED")

# ------------- 6. Prepare the authorised held-out source windows -------------
X_test, y_test, test_metadata, test_overview = prepare_harth_window_cohort(
    subjects=list(RESERVED_PEOPLE),
    data_dir=root / "data/raw/harth_v2",
    task_classes=tuple(SOURCE_CLASSES),
    max_windows_per_subject=approved["source"]["test_cap_per_person"],
    window_size=approved["source"]["window_size"],
    step_size=approved["source"]["step_size"],
    sampling_rate=approved["source"]["sampling_rate_hz"],
)
assert X_test.ndim == 3 and X_test.shape[1:] == (6, 100)
assert len(y_test) == len(test_metadata) and len(y_test) <= 1000
assert set(test_metadata["subject_id"]) == set(RESERVED_PEOPLE)
assert not set(test_metadata["subject_id"]).intersection(DEVELOPMENT)
assert all(test_overview["sampled_windows"].between(1, 500))
assert np.isfinite(X_test).all()

# ------------- 7. Score both frozen models without Test fitting --------------
pred_resnet, pred_rf = infer_source_windows(
    X_test, frozen_nn, frozen_rf, frozen_config, device="cpu"
)
predictions = {
    "1D-ResNet Weighted CE": pred_resnet,
    "Random Forest": pred_rf,
}
y_checked, subject_checked, predictions, code_order = validate_final_test_inputs(
    y_test, test_metadata["subject_id"].to_numpy(), predictions
)
display(test_overview[["subject_id", "eligible_windows", "sampled_windows"]])
print("Qualified Test windows:", len(y_checked))
print("Models evaluated: both frozen Development models")
print("Test-driven training, calibration, threshold tuning: NONE")

**12. Frozen Qualified Test Results**

Report both predeclared models on identical source windows, with fixed eight-class Macro F1, observed-class balanced accuracy, class-specific precision/recall/F1/support, participant-level scores and confusion matrices. Keep absent activities explicit in the class report.

These descriptive results come from only two previously inspected, same-study participants; the selected primary model remains 1D-ResNet with weighted Cross-Entropy regardless of the RF comparison.

In [ ]:
# ===================== 12. Frozen Qualified Test Results =====================

# --------------- 1. Calculate the predefined held-out metrics ----------------
report = summarise_qualified_heldout(
    y_checked, subject_checked, predictions, ACTIVITY_NAMES, SOURCE_CLASSES
)
print("FINAL QUALIFIED SAME-STUDY HELD-OUT EVALUATION")
print("Primary model selected on Development: 1D-ResNet Weighted CE")
print("The Test comparison DOES NOT select a replacement.")
display(report["summary"].round(4))
display(report["per_subject"].round(4))
display(report["per_class"].round(4))

# ---------------- 2. Import plotting for Test result figures -----------------
import matplotlib.pyplot as plt

# ------------------ 3. Plot eight-class confusion matrices -------------------
for model_name in FINAL_MODELS:
    cm = report["confusion"][model_name]
    fig, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_xticks(range(8), [str(code) for code in SOURCE_CLASSES])
    ax.set_yticks(range(8), [str(code) for code in SOURCE_CLASSES])
    ax.set_xlabel("Predicted original source activity code")
    ax.set_ylabel("True original source activity code")
    ax.set_title(f"Qualified held-out count confusion matrix | {model_name}")
    for row in range(8):
        for column in range(8):
            ax.text(column, row, str(int(cm[row, column])),
                    ha="center", va="center", fontsize=8,
                    color="white" if cm[row, column] > 0.5 * cm.max() else "black")
    fig.colorbar(image, ax=ax, label="Overlapping-window count")
    fig.tight_layout()
    plt.show()

# ----------------- 4. Plot class recall with actual support ------------------
for model_name in FINAL_MODELS:
    subset = report["per_class"].query("model == @model_name").copy()
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(subset["activity"], subset["recall"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Recall (0–1)")
    ax.set_title(f"Qualified held-out per-class recall | {model_name}")
    ax.tick_params(axis="x", rotation=40)
    for idx, row in enumerate(subset.itertuples()):
        ax.text(idx, min(row.recall + 0.02, 0.98), f"n={row.support}",
                ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    plt.show()

**13. Final Qualified Evaluation Gate**

Check the approved manifest, frozen artefact hashes, both reserved participant identities, aligned source rows and saved class- and person-level reports. This gate verifies the reproducibility of the qualified evaluation, not independent generalisation, statistical significance or deployment readiness.

Local report files are Git-ignored and need separate retention if the Colab runtime is reset. Stage 27 consolidates this evidence while retaining the historical Test-inspection limitation.

In [ ]:
# ==================== 13. Final Qualified Evaluation Gate ====================

# --------------- 1. Verify approved source and model artefacts ---------------
assert approved["status"] == "APPROVED"
assert set(subject_checked) == set(RESERVED_PEOPLE)
assert len(y_checked) == len(predictions["1D-ResNet Weighted CE"])
assert len(y_checked) == len(predictions["Random Forest"])
assert report["summary"]["model"].tolist() == list(FINAL_MODELS)
assert len(report["per_class"]) == 16 and len(report["per_subject"]) == 4
assert all(report["per_class"].groupby("model")["support"].sum() == len(y_checked))
assert all(int(cm.sum()) == len(y_checked) for cm in report["confusion"].values())
assert all(sha256_file(artifact_info[key]) == artifact_info["sha256"][key]
           for key in ("resnet", "reference", "config"))

# ------------------- 2. Import hashing and path utilities --------------------
import hashlib
from pathlib import Path

# ------------- 3. Export aligned predictions and evidence files --------------
result_dir = root / "outputs/harth_stage26_qualified_test"
result_dir.mkdir(parents=True, exist_ok=True)
report["summary"].to_csv(result_dir / "qualified_model_summary.csv", index=False)
report["per_subject"].to_csv(result_dir / "qualified_per_subject.csv", index=False)
report["per_class"].to_csv(result_dir / "qualified_per_class.csv", index=False)
for model_name in FINAL_MODELS:
    safe_name = "resnet" if model_name.startswith("1D-") else "random_forest"
    pd.DataFrame(report["confusion"][model_name]).to_csv(
        result_dir / f"{safe_name}_confusion_counts.csv", index=False, header=False
    )
pd.DataFrame({
    "subject_id": subject_checked,
    "true_source_code": y_checked,
    "resnet_predicted_source_code": predictions["1D-ResNet Weighted CE"],
    "random_forest_predicted_source_code": predictions["Random Forest"],
}).to_csv(result_dir / "aligned_frozen_predictions.csv", index=False)
provenance = {
    "scientific_status": "qualified same-study held-out; historically inspected; not blind/external",
    "source_revision": approved["source"]["revision"],
    "development_cohort_sha256": approved["source_development_sha256"],
    "approved_manifest_sha256": sha256_file(approved_path),
    "artifact_sha256": {key: sha256_file(artifact_info[key])
                        for key in ("resnet", "reference", "config")},
    "models": list(FINAL_MODELS),
    "test_people": list(RESERVED_PEOPLE),
    "test_windows": len(y_checked),
    "test_tuning_or_retraining": False,
    "fixed_macro_f1_denominator": list(SOURCE_CLASSES),
    "balanced_accuracy_denominator": "Observed true classes (sklearn)",
}
(result_dir / "qualified_test_provenance.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)
print("=" * 76)
print("STAGE 26 | QUALIFIED HELD-OUT EVALUATION: PASSED")
print("=" * 76)
print("Frozen original Test subjects:", ", ".join(RESERVED_PEOPLE))
print("Reported windows:", len(y_checked))
print("Both locked models evaluated on identical source windows: YES")
print("Eight-class Macro F1, per-class and per-person metrics: SAVED")
print("Artefact SHA256 verified after inference: YES")
print("Test-driven model selection/tuning: NO")
print("External validation or independent blind study: NO")
print("Results stored locally:", result_dir)
print("Next: inspect real results, save executed notebook, then Stage 27.")
print("=" * 76)